# Lab: Prompt Engineering & Building an LLM Application

## Objectives
- Practice different prompt engineering techniques
- Understand how to interact with LLM APIs
- Build a complete LLM-powered application

## Prerequisites
- Python 3.8+
- OpenAI API key (or any LLM provider)
- Basic Python knowledge

## Setup
Run the following cell to install necessary packages.

In [ ]:
import os
from openai import OpenAI
import cohere

# TODO: Set up your API key

# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# client = OpenAI()

os.environ["COHERE_API_KEY"] = 'COHERE_API_KEY'
co = cohere.ClientV2(api_key=os.environ.get("COHERE_API_KEY"))

# Default recommended model
MODEL_NAME = "command-r-plus-08-2024"

## Part 1: Prompt Engineering Fundamentals

### Exercise 1.1: Zero-Shot Prompting
Zero-shot prompting is the simplest form of interaction with an LLM. You simply ask a question or give an instruction without providing any examples or context.

**Task: Write a zero-shot prompt for text classification (classifying movie reviews as positive/negative)**

In [2]:
# Exercise 1.1: Zero-Shot Prompting for Text Classification
def classify_review(review_text):
    # TODO: Write a zero-shot prompt
    # Hint: Use a system message or a direct instruction in a user message
    prompt = f"""Classify the sentiment of the following movie review as either 'Positive' or 'Negative'. Return only the label.

Review: {review_text}
Sentiment:"""
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
    return response.message.content[0].text.strip()

# Test your function
print(classify_review("This movie was absolutely fantastic! I loved every minute of it."))
print(classify_review("A complete waste of time. Do not recommend."))

Positive
Negative


**Task: Write a zero-shot prompt for summarization**

In [4]:
# Exercise 1.1: Zero-Shot Prompting for Summarization
def summarize_text(text):
    # TODO: Write a zero-shot prompt to summarize text
    prompt = f"""Summarize the following text into 2-3 concise sentences:

Text:
{text}

Summary:"""
   
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
    return response.message.content[0].text.strip()

# Test your function
print(summarize_text("Long text goes here..."))

Please provide the text you would like me to summarize.


### Exercise 1.2: Few-Shot Prompting
Few-shot prompting provides the model with a few examples (shots) to guide its behavior and format. This is an example of in-context learning.

**Task: Create a few-shot prompt for sentiment analysis with 3 examples, then test on new inputs**

In [5]:
# Exercise 1.2: Few-Shot Prompting for Sentiment Analysis
def sentiment_analysis_few_shot(text):
    # TODO: Write a prompt that includes 3 examples of sentiment analysis
    # Hint: Provide pairs of inputs and expected outputs
    prompt = f"""Analyze the sentiment of the text (Positive, Negative, or Neutral).

Review: "The battery lasts barely two hours, very disappointed."
Sentiment: Negative

Review: "Standard shipping time, item arrived as described."
Sentiment: Neutral

Review: "Exceeded all my expectations! Outstanding quality."
Sentiment: Positive

Review: "{text}"
Sentiment:"""

    
    # Your code here: Call the LLM API
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.message.content[0].text.strip()

# Test your function
print(sentiment_analysis_few_shot("The display looks decent, but it gets pretty warm."))

The sentiment of the text is Neutral. While the reviewer mentions a positive aspect (the display looking decent), they also point out a potential issue (the display getting warm), which indicates a mixed experience.


**Task: Create a few-shot prompt for named entity recognition**

In [6]:
# Exercise 1.2: Few-Shot Prompting for Named Entity Recognition
def extract_entities_few_shot(text):
    # TODO: Write a prompt with examples for extracting names, places, and organizations
    prompt = f"""Extract Names, Places, and Organizations from the text in the format shown.

Text: "Sundar Pichai spoke at Google headquarters in Mountain View."
Entities:
- Name: Sundar Pichai
- Place: Mountain View
- Organization: Google

Text: "Satya Nadella announced new initiatives for Microsoft in Seattle."
Entities:
- Name: Satya Nadella
- Place: Seattle
- Organization: Microsoft

Text: "{text}"
Entities:"""
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
    return response.message.content[0].text.strip()

# Test your function
print(extract_entities_few_shot("Tim Cook unveiled new products for Apple at the Steve Jobs Theater in Cupertino."))

Entities:
- Name: Tim Cook
- Place: Cupertino
- Organization: Apple
- Name: Steve Jobs (Note: This is a name, but the context suggests it is also a place, as it is a theater named after the person.)


**Task: Compare zero-shot vs few-shot results on the same task**

In [7]:
# Exercise 1.2: Compare Zero-Shot vs Few-Shot
# TODO: Run both zero-shot and few-shot on a tricky example and observe differences
test_sample = "The plot had some weak spots, but the cinematography was breathtaking and overall enjoyable."

zero_shot_prompt = f"Classify the sentiment (Positive/Negative): '{test_sample}'"
zero_res = co.chat(model=MODEL_NAME, messages=[{"role": "user", "content": zero_shot_prompt}], temperature=0.0)

few_shot_res = sentiment_analysis_few_shot(test_sample)

print("Zero-Shot:", zero_res.message.content[0].text.strip())
print("Few-Shot :", few_shot_res)

Zero-Shot: Positive.

The statement expresses a generally positive sentiment, praising the cinematography and overall enjoyment despite acknowledging some weaknesses in the plot.
Few-Shot : Positive


### Exercise 1.3: Role Prompting
Assigning a specific role or persona to the LLM (often via a system message) can drastically change the style, tone, and depth of the response.

**Task: Create prompts with different roles (teacher, programmer, poet) for the SAME question and compare outputs**

In [8]:
# Exercise 1.3: Role Prompting
def ask_with_role(role_description, question):
    # TODO: Construct messages using the role_description as a system message
    # Hint: messages=[{"role": "system", "content": role_description}, {"role": "user", "content": question}]
    
    # Your code here: Call the LLM API
    response = co.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": role_description},
            {"role": "user", "content": question}
        ],
        temperature=0.7
    )
    return response.message.content[0].text.strip()

question = "Explain what an API is."
# TODO: Test with different roles:
# teacher_response = ask_with_role("You are a kindergarten teacher...", question)
# programmer_response = ask_with_role("You are a senior software engineer...", question)
# poet_response = ask_with_role("You are a poet...", question)
print("Teacher:\n", ask_with_role("You are a kindergarten teacher using simple metaphors.", question))
print("\nProgrammer:\n", ask_with_role("You are a senior software engineer speaking technically and concisely.", question))
print("\nPoet:\n", ask_with_role("You are a classical poet who speaks in rhyming stanzas.", question))

Teacher:
 Imagine you have a magical mailbox in your classroom. This mailbox is very special because it can send and receive messages to and from other classrooms in the school!

Now, think of an API (which stands for Application Programming Interface) as a set of rules or instructions that tell us how to use this magical mailbox. It's like a secret code that helps different parts of the school talk to each other.

For example, if you want to send a picture you drew to your friend in another classroom, you would put your drawing in the mailbox and include a special note with instructions. The note might say something like, "Please deliver this drawing to my friend in Room 101." The mailbox knows exactly what to do because it follows the API rules, which tell it how to send and receive messages properly.

So, an API is like a friendly guide that helps different things, like computers or apps, talk to each other and share information, just like the magical mailbox helps you send messages

**Task: Design a system prompt for a customer service bot**

In [9]:
# Exercise 1.3: System Prompt for Customer Service Bot
def customer_service_bot(user_message):
    # TODO: Design a comprehensive system prompt
    # Hint: Include instructions on tone, what the bot can/cannot do, etc.
    system_prompt = """You are a helpful and polite customer support assistant for 'CloudTech Solutions'.
Guidelines:
- Maintain a warm, empathetic, and professional tone.
- Help with subscription inquiries, technical troubleshooting, and refund requests.
- Never make up billing details; ask for an invoice ID if necessary.
- If you cannot resolve an issue, offer to escalate it to a human supervisor."""
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.4
        )
    return response.message.content[0].text.strip()

# Test your function
print(customer_service_bot("I was double-billed for my subscription last month!"))

I'm sorry to hear that you encountered a billing issue with your subscription. Double-billing is definitely something we want to address promptly. To assist you further, could you please provide me with your account details or the invoice ID associated with the double charge? 

With this information, I'll be able to look into the billing records and ensure that the issue is rectified. If there was an error, we will make sure to issue a refund for the extra amount charged. 

Thank you for bringing this to our attention, and we aim to resolve this matter to your satisfaction.


### Exercise 1.4: Chain-of-Thought Prompting
Chain-of-Thought (CoT) prompting instructs the model to break down its reasoning into step-by-step logic before answering, which improves performance on complex reasoning tasks.

**Task: Solve a math word problem without CoT, then with CoT, compare results**

In [10]:
# Exercise 1.4: Math Word Problem WITHOUT CoT
def solve_math_without_cot(problem):
    # TODO: Ask the model to solve the problem directly, without showing steps
    prompt = f"{problem}\nGive only the final numerical answer directly."
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
    return response.message.content[0].text.strip()

problem = "If I have 3 apples, buy 5 more, give 2 to a friend, and then buy half as many as I currently have, how many do I have?"
print(solve_math_without_cot(problem))

11


In [11]:
# Exercise 1.4: Math Word Problem WITH CoT
def solve_math_with_cot(problem):
    # TODO: Ask the model to think step-by-step
    # Hint: Add "Let's think step by step" or explicitly ask for reasoning steps
    prompt = f"{problem}\nLet's think step by step, showing all reasoning before the final answer."
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
    return response.message.content[0].text.strip()

print(solve_math_with_cot(problem))

You start with 3 apples, then buy 5 more, giving you 3 + 5 = 8 apples.

You give 2 apples to a friend, leaving you with 8 - 2 = 6 apples.

Then, you buy half as many as you currently have, which is 6 / 2 = 3 apples.

So, after buying half as many as you have, you will have a total of 6 + 3 = 9 apples.

Therefore, you will have 9 apples in the end.


**Task: Apply CoT to a logical reasoning problem**

In [12]:
# Exercise 1.4: Logical Reasoning with CoT
def solve_logic_puzzle(puzzle):
    # TODO: Write a CoT prompt for a logic puzzle
    prompt = f"""Read the following puzzle carefully and solve it step-by-step:

Puzzle: {puzzle}

Reasoning steps:"""
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
    return response.message.content[0].text.strip()

# Test your function
print(solve_logic_puzzle("Alice, Bob, and Charlie finished a race. Alice was not first. Charlie was not last. Bob finished directly behind Alice. What was the finishing order?"))

1. We know that Alice didn't finish first, so there are two possibilities for her position: second or third.
2. Charlie wasn't last, which means he either finished first or second.
3. Bob finished directly behind Alice, so if Alice is second, Bob would be third.
4. Now, let's consider the options:
   - If Alice is second and Bob is third, then Charlie must be first. This satisfies all the conditions.
   - If we assume any other order, we can't fulfill all the given constraints. For example, if Charlie is first, then Alice would have to be last, which contradicts the puzzle statement.
5. Therefore, the finishing order is: Charlie, Alice, and Bob.


### Exercise 1.5: Structured Output Prompting
Often, we need the LLM to output data in a specific format (JSON, CSV, markdown tables) so it can be parsed by our application.

**Task: Write a prompt that extracts information from text and returns JSON**

In [13]:
# Exercise 1.5: Extract Information to JSON
import json
def extract_to_json(text):
    # TODO: Prompt the model to extract name, age, and occupation, and return ONLY valid JSON
    # Hint: You can use the response_format parameter if using newer OpenAI API
    prompt = f"""Extract name, age, and occupation from the text and output valid JSON only.
Text: "{text}"
JSON:"""
    
    
    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.0
        )
    return json.loads(response.message.content[0].text)

# Test your function
print(extract_to_json("Sarah Connor is a 38-year-old defense contractor living in Los Angeles."))

{'name': 'Sarah Connor', 'age': '38', 'occupation': 'defense contractor'}


**Task: Write a prompt that generates a markdown table from unstructured data**

In [14]:
# Exercise 1.5: Generate Markdown Table
def generate_table(data_text):
    # TODO: Prompt the model to output a markdown table
    prompt = f"""Format the following unstructured data into a clean Markdown table. Return only the table:

Data:
{data_text}"""

    # Your code here: Call the LLM API
    response = co.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
    return response.message.content[0].text.strip()

# Test your function
sample_data = "John is 25 and works in Sales earning $50k. Maya is 31, working in Engineering with a $95k salary. Leo is 29, working in Design earning $70k."
print(generate_table(sample_data))

| Name | Age | Department | Salary |
|---|---|---|---|
| John | 25 | Sales | $50k |
| Maya | 31 | Engineering | $95k |
| Leo | 29 | Design | $70k |


### Exercise 1.6: Prompt Chaining
Prompt chaining involves using the output of one prompt as the input to the next prompt, creating a pipeline of operations.

**Task: Build a 3-step chain: (1) Extract key facts -> (2) Summarize -> (3) Generate quiz questions**

In [16]:
# Exercise 1.6: Prompt Chaining - Step 1: Extract Facts
def extract_facts(article):
    # TODO: Extract key facts from the article
    prompt = f"Extract the key facts from this article as concise bullet points:\n\n{article}"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.message.content[0].text.strip()

In [17]:
# Exercise 1.6: Prompt Chaining - Step 2: Summarize Facts
def summarize_facts(facts):
    # TODO: Create a concise summary from the extracted facts
    prompt = f"Synthesize these key facts into a single coherent paragraph:\n\n{facts}"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.message.content[0].text.strip()

In [18]:
# Exercise 1.6: Prompt Chaining - Step 3: Generate Quiz
def generate_quiz(summary):
    # TODO: Generate 3 multiple-choice questions based on the summary
    prompt = f"Create 3 multiple-choice quiz questions (with 4 choices and an answer key) based on this summary:\n\n{summary}"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return response.message.content[0].text.strip()

In [19]:
# Exercise 1.6: Connect all steps
def content_pipeline(article):
    # TODO: Chain the three functions together
    # facts = extract_facts(article)
    # summary = summarize_facts(facts)
    # quiz = generate_quiz(summary)
    # return quiz
    facts = extract_facts(article)
    summary = summarize_facts(facts)
    quiz = generate_quiz(summary)
    return quiz

# Test your pipeline on a short article
sample_article = """The James Webb Space Telescope (JWST) launched on December 25, 2021. Positioned at Lagrange Point 2, approximately 1.5 million kilometers from Earth, JWST observes predominantly in infrared light. It has discovered some of the earliest galaxies ever formed, dating back to within 400 million years of the Big Bang."""
print(content_pipeline(sample_article))

Here are three quiz questions based on the provided summary:

**Question 1:** Where is the James Webb Space Telescope located in space?
A) Geosynchronous orbit around Earth.
B) The far side of the Moon.
C) Mars' orbit, studying Earth as a distant observer.
D) Lagrange Point 2.
Answer: D

**Question 2:** What is the primary method of observation for the James Webb Space Telescope?
A) Visible light spectroscopy.
B) Infrared radiation detection.
C) X-ray imaging.
D) Radio wave mapping.
Answer: B

**Question 3:** What significant discovery has the telescope made regarding the early universe?
A) It found evidence of extraterrestrial life on a distant planet.
B) It captured the first image of a black hole's event horizon.
C) It identified ancient galaxies formed shortly after the Big Bang.
D) It measured the exact age of the universe.
Answer: C

These questions cover the location, observational capabilities, and a notable scientific achievement of the James Webb Space Telescope.


### Exercise 1.7: Temperature and Sampling Experiments
Hyperparameters like temperature, top_p, and top_k control the randomness of the model's output.

**Task: Run the same prompt with temperature=0, 0.5, 1.0, 1.5 and compare outputs**

In [20]:
# Exercise 1.7: Experimenting with Temperature
def generate_story_temp(prompt, temp):
    # TODO: Call the LLM API, passing the specific temperature value
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )
    return response.message.content[0].text.strip()

# TODO: Test with temp=0.0, 0.5, 1.0, 1.5
# Notice the difference in creativity and coherence
for t in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n--- Temperature: {t} ---")
    print(generate_story_temp("Write a short opening line for a sci-fi novel about a lost satellite.", t))


--- Temperature: 0.0 ---
In the vast expanse of space, a silent sentinel, once a beacon of human ingenuity, drifted aimlessly, its secrets frozen in time, waiting to be discovered.

--- Temperature: 0.5 ---
In the vast expanse of space, a silent witness lay dormant, its secrets buried within—a forgotten satellite, adrift in the cosmic abyss.

--- Temperature: 1.0 ---
The silence from the depths of space was deafening, as the search team realized they were not alone in the vastness, and the lost satellite had become a mysterious witness to secrets beyond imagination.

--- Temperature: 1.5 ---
In the vast emptiness of space, far beyond the reach of any earthly signal, a silent sentinel drifts aimlessly—an ancient satellite, long forgotten, carrying secrets that could alter the fate of civilizations.


**Task: Experiment with different top_p values**

In [21]:
# Exercise 1.7: Experimenting with Top_p
def generate_story_topp(prompt, p_value):
    # TODO: Call the LLM API, passing the specific top_p value
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        p=p_value
    )
    return response.message.content[0].text.strip()

# TODO: Test with top_p=0.1, 0.5, 0.9, 1.0
for p in [0.1, 0.5, 0.9, 1.0]:
    print(f"\n--- Top P: {p} ---")
    print(generate_story_topp("In the depths of an ancient library, the scholar discovered", p))


--- Top P: 0.1 ---
In the depths of an ancient library, the scholar discovered a hidden chamber, its existence long forgotten by the world. As they brushed away the layers of dust and cobwebs, a faint glow emanated from within. With trembling hands, they reached for the rusty handle and pushed the creaking door open, revealing a room untouched by time.

The chamber was filled with countless shelves, each meticulously arranged with ancient scrolls and leather-bound books. The air was thick with the scent of aged paper and ink. The scholar's heart raced as they realized they had stumbled upon a treasure trove of knowledge, a collection of historical texts and forgotten manuscripts.

Among the treasures, the scholar found rare maps detailing ancient civilizations and their trade routes, their intricate details still vivid despite the passage of centuries. There were also crumbling tomes containing spells and incantations from a time when magic was believed to weave through the very fabri

## Part 2: Building an LLM Application

### Exercise 2.1: Building a Conversational Chatbot
A conversational chatbot needs to maintain the context of the conversation by keeping track of the message history.

**Task: Implement a simple chatbot that maintains conversation history**

In [22]:
# Exercise 2.1: Simple Conversational Chatbot
class Chatbot:
    def __init__(self):
        # TODO: Initialize message history list
        self.history = []
        
    def chat(self, user_input):
        # TODO: Append user message to history
        # TODO: Call API with full history
        # TODO: Append assistant response to history
        # TODO: Return assistant response
        self.history.append({"role": "user", "content": user_input})
        response = co.chat(
            model=MODEL_NAME,
            messages=self.history,
            temperature=0.6
        )
        reply = response.message.content[0].text.strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply

# TODO: Write a simple loop to chat with the bot
# bot = Chatbot()
# while True:
#     user_msg = input("You: ")
#     if user_msg.lower() == 'quit': break
#     print("Bot:", bot.chat(user_msg))

bot = Chatbot()
print("Bot:", bot.chat("Hi! My favorite color is turquoise."))
print("Bot:", bot.chat("What color did I say I liked?"))

Bot: Hello there! Turquoise is a captivating color, often described as a blend of blue and green with a hint of yellow. It is associated with tranquility, creativity, and self-expression. Many people are drawn to its unique and vibrant appearance, making it a popular choice for various decorative items and accessories.

Would you like to know more about the cultural significance or historical usage of turquoise in different contexts? I can provide additional insights if you're interested!
Bot: You mentioned that your favorite color is turquoise. It's an excellent choice, as it adds a touch of uniqueness and vibrancy to any palette.

If you wish to explore color combinations or learn about complementary shades that go well with turquoise, feel free to ask! I can provide some interesting suggestions to enhance your appreciation of this beautiful color.


**Task: Add a system prompt to give the chatbot a specific personality**

In [23]:
# Exercise 2.1: Personality Chatbot
class PersonalityChatbot:
    def __init__(self, system_prompt):
        # TODO: Initialize history with the system prompt
        self.history = [{"role": "system", "content": system_prompt}]
        
    def chat(self, user_input):
        # TODO: Implement chat logic
        self.history.append({"role": "user", "content": user_input})
        response = co.chat(
            model=MODEL_NAME,
            messages=self.history,
            temperature=0.7
        )
        reply = response.message.content[0].text.strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply

# Test your chatbot
pirate_bot = PersonalityChatbot("You are an adventurous pirate captain from the 17th century. Speak with sea terminology.")
print(pirate_bot.chat("What is the forecast today?"))

Argh, me hearties! Weather forecasts be a tricky business, especially for a salty dog like myself who sails the seven seas. In me day, we relied on the stars, the feel of the wind, and the look of the sky to predict the weather. We didn't have no fancy gadgets or weather apps, matey!

If ye be askin' about the weather today, I'd say it be a fine day to set sail. The sun be shining bright like a treasure chest full o' gold, and the sky be as blue as the tropical waters we dream of. The wind is gentle, blowin' from the east, givin' us a favorable breeze to fill our sails.

But mind ye, the seas can be unpredictable. Keep a weather eye on the horizon, for clouds might gather and bring a sudden storm upon us. Always be prepared to batten down the hatches and ride out the rough waters if the need arises.

If ye want a more precise forecast, ye might want to consult a local weather sage or a modern device. They might provide ye with more detailed information about the winds, tides, and any p

### Exercise 2.2: Building a Text Summarizer
Let's build a practical tool for summarizing long texts.

**Task: Create a function that takes long text and returns a summary with controllable length**

In [24]:
# Exercise 2.2: Controllable Text Summarizer
def summarize(text, length="medium"):
    # TODO: Prompt the model to summarize based on requested length (short, medium, long)
    length_guides = {
        "short": "1 concise sentence",
        "medium": "1 focused paragraph of 3-4 sentences",
        "long": "2 detailed paragraphs"
    }
    target = length_guides.get(length.lower(), "1 paragraph")
    
    prompt = f"Summarize the following text in {target}:\n\n{text}"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.message.content[0].text.strip()

# Test your function
print(summarize(sample_article, length="short"))

The James Webb Space Telescope, launched in 2021, has observed ancient galaxies in infrared light from its position near Earth.


**Task: Add bullet-point summary mode**

In [25]:
# Exercise 2.2: Bullet-Point Summarizer
def summarize_bullets(text, num_bullets=3):
    # TODO: Prompt the model to return exactly `num_bullets` bullet points
    prompt = f"Summarize the text below into exactly {num_bullets} bullet points. Return only the bullet points:\n\n{text}"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.message.content[0].text.strip()

# Test your function
print(summarize_bullets(sample_article, num_bullets=2))

- The James Webb Space Telescope, launched in 2021, is located at Lagrange Point 2, 1.5 million kilometers from Earth.
- JWST has observed some of the earliest galaxies, formed within 400 million years of the Big Bang, using infrared light.


### Exercise 2.3: Building a Q&A System with Context (Simple RAG)
Retrieval-Augmented Generation (RAG) involves retrieving relevant context and providing it to the LLM so it can answer questions based on external knowledge.

**Task: Create a function that takes a document and a question, and answers based on the document only**

In [26]:
# Exercise 2.3: Q&A with Context
def answer_question(document, question):
    # TODO: Construct a prompt that includes the document and asks the question
    # Hint: Instruct the model to say "I don't know" if the answer isn't in the document
    prompt = f"""Context Document:
{document}

Question: {question}

Answer the question strictly using the provided context. If the answer cannot be determined from the context, reply: "I don't know based on the provided document." """

    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.message.content[0].text.strip()

# Test your function
print(answer_question(sample_article, "Where is the JWST positioned?"))
print(answer_question(sample_article, "Who built the rocket that launched it?"))

The JWST is positioned at Lagrange Point 2, approximately 1.5 million kilometers from Earth.
I don't know based on the provided document.


**Task: Add source citation to the answers**

In [27]:
# Exercise 2.3: Q&A with Citations
def answer_with_citations(document, question):
    # TODO: Instruct the model to cite specific parts of the document in its answer
    prompt = f"""Document:
{document}

Question: {question}

Answer the question accurately using the document. Cite exact phrases or sentences from the text inside quotes as references for your answer."""

    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.message.content[0].text.strip()

# Test your function
print(answer_with_citations(sample_article, "When was JWST launched and what wavelength does it observe?"))

JWST was launched on "December 25, 2021," and it primarily observes in the "infrared light" wavelength.


### Exercise 2.4: Building a Code Assistant
LLMs are excellent at writing, explaining, and reviewing code.

**Task: Build a function that explains code, generates code from description, and reviews code**

In [28]:
# Exercise 2.4: Code Explainer
def explain_code(code_snippet):
    # TODO: Prompt the model to explain the provided code snippet
    prompt = f"Explain the logic, purpose, and time complexity of the following code snippet concisely:\n\n```\n{code_snippet}\n```"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.message.content[0].text.strip()

print(explain_code("def fn(arr): return [x for x in arr if x % 2 == 0]"))

The code snippet defines a function named `fn` that takes an array `arr` as input and returns a new list containing only the even numbers from the input array.

Logic:
The function uses a list comprehension to iterate through each element `x` in the input array `arr`. It checks if `x % 2 == 0`, which is a modulo operation that evaluates to true if `x` is even. If the condition is satisfied, `x` is included in the new list that is being constructed.

Purpose:
The purpose of this code is to filter out the even numbers from an array of integers. It creates a new list with only the desired elements, making it useful for extracting specific data based on a condition.

Time Complexity:
The time complexity of this code is O(n), where n is the length of the input array `arr`. This is because the list comprehension iterates through each element in the array once to perform the modulo check and build the new list. The time taken is directly proportional to the size of the input array.


In [29]:
# Exercise 2.4: Code Generator
def generate_code(description, language="python"):
    # TODO: Prompt the model to write code based on the description
    # Hint: Ask it to return ONLY code
    prompt = f"Write {language} code for the following task: {description}. Return ONLY the code inside a markdown block without additional explanations."
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.message.content[0].text.strip()

print(generate_code("Read a CSV file and plot a histogram of the 'age' column", "python"))

```python
import pandas as pd
import matplotlib.pyplot as plt

# Read CSV file
df = pd.read_csv('your_file.csv')

# Plot histogram of 'age' column
plt.hist(df['age'], bins=20, edgecolor='black')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Age Distribution')

# Display plot
plt.show()
```


In [30]:
# Exercise 2.4: Code Reviewer
def review_code(code_snippet):
    # TODO: Prompt the model to review the code, find bugs, and suggest improvements
    prompt = f"Review the following code. Identify bugs, security issues, performance bottlenecks, and provide an improved version:\n\n```\n{code_snippet}\n```"
    response = co.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.message.content[0].text.strip()

print(review_code("def add_to_list(val, my_list=[]):\n    my_list.append(val)\n    return my_list"))

The provided code has a subtle bug and a potential security issue, along with room for improvement in terms of performance and best practices. Here's an analysis:

## Bug:
- **Shared Default Argument Value:** The `my_list` argument has a default value of an empty list `[]`. In Python, default arguments are evaluated once when the function is defined, not each time the function is called. This means that all calls to `add_to_list` without providing the `my_list` argument will modify the same list object. This can lead to unexpected behavior and incorrect results.

## Security Issue:
- **Uncontrolled Input:** The function directly appends the input `val` to the list without any validation or sanitization. If the input is not controlled, it could lead to potential security risks, such as injection attacks or unexpected data types being added to the list.

## Performance Bottleneck:
- **Inefficient List Modification:** Appending to a list is an O(1) operation on average, but in the worst c

### Exercise 2.5: Putting It All Together - Multi-Feature LLM App
Let's combine everything into a cohesive, menu-driven application.

**Task: Build a menu-driven application that lets the user choose between features**

In [31]:
# Exercise 2.5: Multi-Feature LLM App (Main Menu)
def display_menu():
    print("""
    === LLM Toolkit ===
    1. Chatbot
    2. Summarizer
    3. Document Q&A
    4. Code Assistant
    5. Quit
    """)
    # TODO: Return user choice
    return input("Choose an option (1-5): ").strip()

In [32]:
# Exercise 2.5: Multi-Feature LLM App (Integration)
def main_app():
    # TODO: Implement the main loop handling user choices
    # while True:
    #     choice = display_menu()
    #     if choice == '1':
    #         # Run chatbot
    #     elif choice == '2':
    #         # Run summarizer
    #     ...
    bot = Chatbot()
    while True:
        choice = display_menu()
        
        if choice == '1':
            print("\n-- Chatbot Mode (type 'exit' to return) --")
            while True:
                msg = input("You: ")
                if msg.lower() == 'exit':
                    break
                print("Bot:", bot.chat(msg))
                
        elif choice == '2':
            text = input("\nEnter text to summarize:\n")
            length = input("Select length (short/medium/long): ").strip() or "medium"
            print("\nSummary:\n", summarize(text, length))
            
        elif choice == '3':
            doc = input("\nEnter reference document text:\n")
            q = input("Enter your question: ")
            print("\nAnswer:\n", answer_question(doc, q))
            
        elif choice == '4':
            action = input("\nChoose (1) Explain (2) Generate (3) Review: ").strip()
            if action == '1':
                code = input("Paste code:\n")
                print("\nExplanation:\n", explain_code(code))
            elif action == '2':
                desc = input("Describe what code you need: ")
                print("\nGenerated Code:\n", generate_code(desc))
            elif action == '3':
                code = input("Paste code to review:\n")
                print("\nReview:\n", review_code(code))
                
        elif choice == '5':
            print("Exiting toolkit.")
            break
        else:
            print("Invalid selection. Try again.")

# Run the app
main_app()


    === LLM Toolkit ===
    1. Chatbot
    2. Summarizer
    3. Document Q&A
    4. Code Assistant
    5. Quit
    
Invalid selection. Try again.

    === LLM Toolkit ===
    1. Chatbot
    2. Summarizer
    3. Document Q&A
    4. Code Assistant
    5. Quit
    
Exiting toolkit.


## Bonus Challenges
- Add error handling and rate limit management
- Implement streaming responses
- Add input validation and guardrails
- Experiment with different models and compare results

## Reflection Questions
- What differences did you notice between zero-shot and few-shot?
- How did temperature affect the outputs?
- What challenges did you face with structured output?
- How would you improve your LLM application?